# Week 2 on a GPU - embed the whole catalog

Your laptop runs at about **0.9 songs/sec**, so all 13,497 tracks take
roughly **4.2 hours**. A free Colab T4 does it in **60-90 minutes**.

## Read this bit, it matters

**Colab deletes everything when you disconnect.** Close the lid, lose wifi,
leave it idle - the whole machine is thrown away, including hours of work.
This already happened once on this project and cost ~9,000 embedded songs.

So this notebook saves everything to **your Google Drive** instead of
Colab's temporary disk. A disconnect now costs you nothing: reconnect, run
the cells again, and it picks up exactly where it stopped.

**Before you start:** `Runtime` -> `Change runtime type` -> **T4 GPU** -> Save.

## 1. Check you got a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU - change Runtime type to T4'

## 2. Connect your Google Drive

This is the change that makes the job crash-proof. A pop-up asks you to
choose your Google account and click **Allow**.

Everything gets written to a folder called `music-instagram` in your Drive,
so it survives disconnects, and you can see the files from any device.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/music-instagram'
os.makedirs(DRIVE, exist_ok=True)
print('saving everything to:', DRIVE)
print('files already there:', sorted(os.listdir(DRIVE)) or '(empty - first run)')

## 3. Install what Colab is missing

**PyAV** decodes the previews. They are `.m4a` (AAC) files, not MP3, and
most Python audio libraries cannot read AAC at all.

In [ ]:
!pip install -q av faiss-cpu
import av, torch
print('PyAV', av.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 4. Get the project code

In [ ]:
import os
if os.path.exists('/content/Music-Instagram'):
    !git -C /content/Music-Instagram pull -q
else:
    !git clone -q https://github.com/deepakkarhana/Music-Instagram.git
%cd /content/Music-Instagram
!git log --oneline -1

## 5. Put your catalog on Drive (first run only)

If the catalog is already on your Drive, this cell skips instantly. That is
the point - after a disconnect you do not upload anything again.

On the first run it asks for this file from your laptop:

```
Desktop/Music-Instagram/data/raw/itunes_catalog.csv
```

**Always upload it rather than re-harvesting.** iTunes returns slightly
different results each run, so a fresh harvest would have different track
ids and the embeddings would line up with the wrong songs.

In [ ]:
import os, shutil
catalog = os.path.join(DRIVE, 'itunes_catalog.csv')

if os.path.exists(catalog):
    n = sum(1 for _ in open(catalog, encoding='utf-8')) - 1
    print(f'already on Drive: {n:,} tracks - nothing to upload')
else:
    from google.colab import files
    for name in files.upload():
        shutil.move(name, catalog)
    n = sum(1 for _ in open(catalog, encoding='utf-8')) - 1
    print(f'uploaded {n:,} tracks to Drive')

## 6. Seed with the work your laptop already did (first run only)

Your laptop has **1,560** songs already embedded. Upload them once and
they live on Drive from then on - the next cell picks them up
automatically.

Select **both** files (hold `Ctrl`):

```
Desktop/Music-Instagram/data/interim/clap_audio.f32
Desktop/Music-Instagram/data/interim/clap_audio_ids.csv
```

Skips instantly if they are already on Drive. **Cancel** to start fresh.

In [ ]:
import os, shutil
store_ids = os.path.join(DRIVE, 'clap_audio_ids.csv')

if os.path.exists(store_ids):
    print(f'already on Drive: {sum(1 for _ in open(store_ids)):,} songs embedded')
else:
    from google.colab import files
    print('Upload clap_audio.f32 and clap_audio_ids.csv, or press Cancel to skip.')
    try:
        for name in files.upload():
            shutil.move(name, os.path.join(DRIVE, name))
        print('seeded from your laptop')
    except Exception:
        print('starting fresh')

## 7. Embed everything

**60-90 minutes.** Progress prints every batch with a time estimate.

### How the crash protection works

Embeddings are written to Colab's fast local disk, and **copied to your
Drive every 10 batches** (about every 320 songs). Writing directly to
Drive would be far slower - it is a network drive pretending to be a
disk, and thousands of tiny writes crawl.

So if Colab disconnects, you lose at most the last ~320 songs. Reconnect,
run cells 2 to 7 again, and it restores from Drive and carries on. You
will see `Restored 3 files` when that happens.

Check this line before walking away:

```
Text encoder health check passed (offset 0.6681, want < 0.98)
```

If it says failed, **stop** - the model is broken and this would produce
meaningless numbers.

**Keep the tab open and the laptop awake.**

In [ ]:
!python -u scripts/03_embed_catalog.py --catalog "{DRIVE}/itunes_catalog.csv" --mirror "{DRIVE}" --batch 32 --workers 16

## 8. Build the index and try it

In [ ]:
!python scripts/04_build_index.py --catalog "{DRIVE}/itunes_catalog.csv"

In [ ]:
!python scripts/05_search.py "rainy cafe window, quiet piano, soft melancholy" -k 5 --catalog "{DRIVE}/itunes_catalog.csv"

## 9. Get the files onto your laptop

They are already safe on your Drive. This just downloads them so the project
works offline.

Put both in `Desktop/Music-Instagram/data/interim/`, replacing what is there,
then run `python scripts/04_build_index.py` on your laptop.

If the browser blocks the download, open Drive and get them from the
`music-instagram` folder instead - same files.

In [ ]:
from google.colab import files
files.download(f'{DRIVE}/clap_audio.f32')
files.download(f'{DRIVE}/clap_audio_ids.csv')